In [1]:
"""
FASE 7: Robustez ao início do contato (ignora os primeiros 10 ms da aceitação de peso)
=====================================================================================
VERSÃO AUTÔNOMA: não importa nenhum outro script. Pode ser colada inteira numa célula
do Jupyter.

Motivo: o pico em valgo ocorreu, em mediana, a 7% da aceitação de peso, muitas vezes
no primeiro quadro do apoio, instante que pode conter artefato do início da força de
reação do solo. O script refaz a extração ignorando os primeiros 10 ms após o contato
inicial e repete: rotulagem P75, LOSO (P75 global e P75 do treino), teste de
permutação no nível do participante e comparação entre validação por ciclo e por
participante.

Saídas em PASTA_FASE3: dataset_desfecho_frontal_ignora10ms.xlsx e
fase7_robustez_contato.txt
"""

import glob
import os
import re
import sys
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             matthews_corrcoef, precision_score, recall_score,
                             roc_auc_score)
from sklearn.model_selection import LeaveOneGroupOut, StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC

# ============================================================================
# CONFIGURAÇÕES
# ============================================================================
PASTA_FASE3 = r'C:\JG\opensim\Resultados_FASE3'
PASTA_ZENODO = (r'C:\JG\opensim\Dados_Zenodo'
                r'\Nitschke_et_al_Change_of_Direction_Marker_Simulation')
IGNORAR_INICIO_MS = 10.0
FC_FILTRO = 15.0
LIMIAR_CONTATO = 20.0
N_PERMUTACOES = 1000
SEMENTE = 42
BASE = 'knee_r_on_tibia_r_in_tibia_r_'
ANGULOS = ['Angulo_Aducao_Quadril', 'Angulo_Flexao_Joelho',
           'Angulo_Flexao_Quadril', 'Angulo_Rotacao_Quadril']
POR_MASSA = {'Momento_Aducao_Quadril': 'Momento_Aducao_Quadril_Nm_kg',
             'Cisalhamento_ML': 'Cisalhamento_ML_N_kg',
             'Compressao_Y': 'Compressao_Y_N_kg'}
FEATURES = ANGULOS + list(POR_MASSA.values())


class Tee:
    def __init__(self, caminho):
        self.arquivo = open(caminho, 'w', encoding='utf-8')
        self.tela = sys.stdout

    def write(self, texto):
        self.tela.write(texto)
        self.arquivo.write(texto)

    def flush(self):
        self.tela.flush()
        self.arquivo.flush()


def titulo(texto):
    print('\n' + '=' * 78 + '\n' + texto + '\n' + '=' * 78)


# ============================================================================
# EXTRAÇÃO (igual à FASE 4, com o início do contato ignorado)
# ============================================================================
def ler_storage(caminho):
    em_graus = None
    with open(caminho, 'r', errors='ignore') as f:
        for i, linha in enumerate(f):
            if linha.lower().startswith('indegrees'):
                em_graus = 'yes' in linha.lower()
            if 'endheader' in linha.lower():
                break
    df = pd.read_csv(caminho, sep=r'\s+', skiprows=i + 1, engine='python')
    df.columns = [c.strip() for c in df.columns]
    return df, em_graus


def passa_baixa(sinal, fs, fc=FC_FILTRO, ordem=3):
    b, a = butter(ordem, fc / (fs / 2.0))
    return filtfilt(b, a, sinal)


def processar(pasta, ignorar_ms):
    m = re.search(r'P(\d+)_trial(\d+)', os.path.basename(pasta))
    part, trial = m.group(1), m.group(2)
    jra, _ = ler_storage(os.path.join(pasta, 'JRA_JointReaction_ReactionLoads.sto'))
    id_, _ = ler_storage(os.path.join(pasta, 'inverse_dynamics.sto'))
    grf, _ = ler_storage(os.path.join(pasta, 'grf_medidas_filtradas.mot'))
    arq = glob.glob(os.path.join(PASTA_ZENODO, f'Participant_{part}', '**',
                                 f'trial{trial}_inverse_methods_coordinates.mot'), recursive=True)[0]
    coord, em_graus = ler_storage(arq)
    # Mesmo recorte do fase4: só os quadros dentro da janela dos autores
    arq_mom = glob.glob(os.path.join(PASTA_ZENODO, f'Participant_{part}', '**',
                                     f'trial{trial}_inverse_methods_moments.sto'), recursive=True)[0]
    t0_autores = float(ler_storage(arq_mom)[0]['time'].min())
    jra = jra[jra['time'] >= t0_autores - 1e-6].reset_index(drop=True)
    t = jra['time'].to_numpy()
    fs_c = 1.0 / np.median(np.diff(coord['time']))

    def ang(nome):
        v = passa_baixa(coord[nome].to_numpy(), fs_c)
        if em_graus is False:
            v = np.degrees(v)
        return np.interp(t, coord['time'], v)

    joelho = ang('knee_angle_r')
    apoio = np.interp(t, grf['time'], grf['ground_force_vy']) > LIMIAR_CONTATO
    idx_apoio = np.where(apoio)[0]
    i_ic = idx_apoio[0]
    i_flex = idx_apoio[np.argmin(joelho[idx_apoio])]
    aceit = np.zeros_like(apoio)
    aceit[i_ic:i_flex + 1] = True
    aceit &= apoio & (t >= t[i_ic] + ignorar_ms / 1000.0)
    mx = jra[f'{BASE}mx'].to_numpy()
    idx = np.where(aceit)[0]
    i_p = idx[np.argmax(mx[idx])]
    t_p = t[i_p]
    return {
        'Participante': part, 'Movimento': trial,
        'Momento_Valgo_Pico_Nm': float(mx[i_p]),
        'Pico_pct_Aceitacao': float(100 * (t_p - t[i_ic]) / max(t[i_flex] - t[i_ic], 1e-9)),
        'Angulo_Flexao_Joelho': float(joelho[i_p]),
        'Angulo_Flexao_Quadril': float(ang('hip_flexion_r')[i_p]),
        'Angulo_Aducao_Quadril': float(ang('hip_adduction_r')[i_p]),
        'Angulo_Rotacao_Quadril': float(ang('hip_rotation_r')[i_p]),
        'Momento_Aducao_Quadril': float(np.interp(t_p, id_['time'], id_['hip_adduction_r_moment'])),
        'Cisalhamento_ML': float(jra[f'{BASE}fz'].iloc[i_p]),
        'Compressao_Y': float(-jra[f'{BASE}fy'].iloc[i_p]),
    }


def massa_osim(part):
    arq = glob.glob(os.path.join(PASTA_ZENODO, f'Participant_{part}', '**',
                                 f'Participant_{part}.osim'), recursive=True)[0]
    raiz = ET.parse(arq).getroot()
    return sum(float(b.find('mass').text) for b in raiz.iter('Body')
               if b.find('mass') is not None and b.find('mass').text)


# ============================================================================
# MODELO E MÉTRICAS
# ============================================================================
def novo_modelo():
    return make_pipeline(MinMaxScaler(), SVC(kernel='linear', C=1.0,
                                             class_weight='balanced', random_state=SEMENTE))


def metricas(y, pred, escore):
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {'acuracia': accuracy_score(y, pred),
            'acuracia_balanceada': balanced_accuracy_score(y, pred),
            'sensibilidade': recall_score(y, pred, zero_division=0),
            'especificidade': tn / (tn + fp) if (tn + fp) else np.nan,
            'precisao': precision_score(y, pred, zero_division=0),
            'MCC': matthews_corrcoef(y, pred),
            'AUC': roc_auc_score(y, escore) if len(np.unique(y)) > 1 else np.nan}


def loso(X, d, g, limiar_no_treino=False, detalhar=False):
    y_glob = (d >= np.quantile(d, 0.75)).astype(int)
    y_ref, esc, pred = np.full(len(d), -1), np.full(len(d), np.nan), np.full(len(d), -1)
    linhas = []
    for tr, te in LeaveOneGroupOut().split(X, d, g):
        if limiar_no_treino:
            lim = np.quantile(d[tr], 0.75)
            y_tr, y_te = (d[tr] >= lim).astype(int), (d[te] >= lim).astype(int)
        else:
            y_tr, y_te = y_glob[tr], y_glob[te]
        y_ref[te] = y_te
        if len(np.unique(y_tr)) < 2:
            continue
        m = novo_modelo().fit(X[tr], y_tr)
        esc[te], pred[te] = m.decision_function(X[te]), m.predict(X[te])
        if detalhar:
            tn, fp, fn, tp = confusion_matrix(y_te, pred[te], labels=[0, 1]).ravel()
            linhas.append({'participante_teste': g[te][0], 'C1_treino': int(y_tr.sum()),
                           'VP': tp, 'FN': fn, 'FP': fp, 'VN': tn})
    ok = ~np.isnan(esc)
    return metricas(y_ref[ok], pred[ok], esc[ok]), pd.DataFrame(linhas)


def imprimir(nome, m):
    print(f'\n{nome}')
    for k, v in m.items():
        print(f'  {k:20s}: {v:.3f}')


# ============================================================================
# EXECUÇÃO
# ============================================================================
saida_txt = Tee(os.path.join(PASTA_FASE3, 'fase7_robustez_contato.txt'))
sys.stdout = saida_txt
try:
    pastas = sorted(p for p in glob.glob(os.path.join(PASTA_FASE3, 'P*_trial*'))
                    if os.path.isdir(p)
                    and re.fullmatch(r'P\d+_trial\d+', os.path.basename(p)))   # ignora cópias como *_original
    novo = pd.DataFrame([processar(p, IGNORAR_INICIO_MS) for p in pastas])
    orig = pd.DataFrame([processar(p, 0.0) for p in pastas])
    novo.to_excel(os.path.join(PASTA_FASE3,
                               f'dataset_desfecho_frontal_ignora{IGNORAR_INICIO_MS:g}ms.xlsx'),
                  index=False)

    titulo(f'1. DESFECHO COM E SEM OS PRIMEIROS {IGNORAR_INICIO_MS:g} ms')
    comp = pd.DataFrame({'Participante': novo.Participante, 'Movimento': novo.Movimento,
                         'Valgo_original_Nm': orig.Momento_Valgo_Pico_Nm,
                         'Valgo_novo_Nm': novo.Momento_Valgo_Pico_Nm,
                         'Pico_pct_original': orig.Pico_pct_Aceitacao,
                         'Pico_pct_novo': novo.Pico_pct_Aceitacao})
    print(comp.round(2).to_string(index=False))
    mudou = (comp.Valgo_original_Nm - comp.Valgo_novo_Nm).abs() > 0.01
    print(f'\nCiclos cujo pico mudou: {mudou.sum()}/{len(comp)}')
    print(f'Spearman entre as duas versões do desfecho: '
          f'{comp.Valgo_original_Nm.corr(comp.Valgo_novo_Nm, method="spearman"):.3f}')

    massas = {p: massa_osim(p) for p in novo.Participante.unique()}
    df = novo.copy()
    df['Massa_kg'] = df.Participante.map(massas)
    for o, n in POR_MASSA.items():
        df[n] = df[o] / df.Massa_kg
    d = (df.Momento_Valgo_Pico_Nm / df.Massa_kg).to_numpy()
    X = df[FEATURES].to_numpy(float)
    g = df.Participante.to_numpy()
    y = (d >= np.quantile(d, 0.75)).astype(int)

    titulo('2. ROTULAGEM (P75) COM O DESFECHO NOVO')
    print(f'Limiar P75: {np.quantile(d, 0.75):.3f} Nm/kg | Classe 1: {y.sum()} | Classe 0: {(y == 0).sum()}')
    print('Classe 1 por participante:', pd.Series(y).groupby(g).sum().to_dict())

    titulo('3. LOSO')
    m_g, tab = loso(X, d, g, detalhar=True)
    print(tab.to_string(index=False))
    imprimir('Agregado (P75 global):', m_g)
    imprimir('Agregado (P75 do treino):', loso(X, d, g, limiar_no_treino=True)[0])

    titulo(f'4. PERMUTAÇÃO NO NÍVEL DO PARTICIPANTE ({N_PERMUTACOES})')
    rng = np.random.default_rng(SEMENTE)
    atletas = np.unique(g)
    blocos = {a: d[g == a] for a in atletas}
    nulos = []
    for _ in range(N_PERMUTACOES):
        trocados = rng.permutation(atletas)
        dp = d.copy()
        for a_o, a_n in zip(atletas, trocados):
            dp[g == a_o] = blocos[a_n]
        nulos.append(loso(X, dp, g)[0]['AUC'])
    nulos = np.array(nulos)
    p = (np.sum(nulos >= m_g['AUC']) + 1) / (N_PERMUTACOES + 1)
    print(f'AUC observada {m_g["AUC"]:.3f} | nula mediana {np.median(nulos):.3f} '
          f'[P2,5 {np.percentile(nulos, 2.5):.3f}; P97,5 {np.percentile(nulos, 97.5):.3f}] | p = {p:.3f}')

    titulo('5. POR CICLO x POR PARTICIPANTE (3 partições, 50 repetições)')
    for nome, fab, grp in (
            ('Por ciclo', lambda r: StratifiedKFold(3, shuffle=True, random_state=r), None),
            ('Por participante', lambda r: StratifiedGroupKFold(3, shuffle=True, random_state=r), g)):
        aucs, baccs = [], []
        for r in range(50):
            s, pr = np.full(len(y), np.nan), np.full(len(y), -1)
            div = fab(r)
            for tr, te in (div.split(X, y, grp) if grp is not None else div.split(X, y)):
                if len(np.unique(y[tr])) < 2:      # treino com uma só classe: partição ignorada
                    continue
                mod = novo_modelo().fit(X[tr], y[tr])
                s[te], pr[te] = mod.decision_function(X[te]), mod.predict(X[te])
            ok = ~np.isnan(s)
            if len(np.unique(y[ok])) < 2:
                continue
            aucs.append(roc_auc_score(y[ok], s[ok]))
            baccs.append(balanced_accuracy_score(y[ok], pr[ok]))
        print(f'{nome}: AUC média {np.mean(aucs):.3f} [P2,5 {np.percentile(aucs, 2.5):.3f}; '
              f'P97,5 {np.percentile(aucs, 97.5):.3f}] | acurácia balanceada {np.mean(baccs):.3f}')

    titulo('CONCLUÍDO')
finally:
    sys.stdout = saida_txt.tela
    saida_txt.arquivo.close()


1. DESFECHO COM E SEM OS PRIMEIROS 10 ms
Participante Movimento  Valgo_original_Nm  Valgo_novo_Nm  Pico_pct_original  Pico_pct_novo
          02      0122              10.40          10.40               8.11           8.11
          02      0123              15.38          15.38               9.68           9.68
          02      0130              15.85          15.85              10.00          10.00
          03      0089               5.55           2.10               4.00           8.00
          03      0095               6.86           6.80               3.85           7.69
          03      0099               5.89           5.89               8.00           8.00
          04      0079              11.92           8.97               3.33           6.67
          04      0080              13.71           9.47               3.45           6.90
          04      0081              20.34          20.34               6.45           6.45
          05      0084               7.12       

## Fase7 secao5 apenas 
O erro aconteceu só na última parte, a seção 5. Todo o resto rodou, e os resultados confirmam o quadro. Ele lê o Excel que já foi gerado e refaz só essa comparação, em segundos.

In [2]:
"""
FASE 7, seção 5 apenas: validação por ciclo x por participante, com o desfecho que
ignora os primeiros 10 ms. Lê o Excel já gerado pelo fase7 (não refaz a extração nem
a permutação). Partições cujo treino tem uma só classe são ignoradas.
"""
import glob, os, xml.etree.ElementTree as ET
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC

PASTA_FASE3 = r'C:\JG\opensim\Resultados_FASE3'
PASTA_ZENODO = (r'C:\JG\opensim\Dados_Zenodo'
                r'\Nitschke_et_al_Change_of_Direction_Marker_Simulation')

df = pd.read_excel(os.path.join(PASTA_FASE3, 'dataset_desfecho_frontal_ignora10ms.xlsx'),
                   dtype={'Participante': str})
df['Participante'] = df['Participante'].str.zfill(2)

def massa(p):
    arq = glob.glob(os.path.join(PASTA_ZENODO, f'Participant_{p}', '**', f'Participant_{p}.osim'),
                    recursive=True)[0]
    return sum(float(b.find('mass').text) for b in ET.parse(arq).getroot().iter('Body')
               if b.find('mass') is not None and b.find('mass').text)

df['Massa_kg'] = df.Participante.map({p: massa(p) for p in df.Participante.unique()})
for o in ('Momento_Aducao_Quadril', 'Cisalhamento_ML', 'Compressao_Y'):
    df[o + '_kg'] = df[o] / df.Massa_kg
feats = ['Angulo_Aducao_Quadril', 'Angulo_Flexao_Joelho', 'Angulo_Flexao_Quadril',
         'Angulo_Rotacao_Quadril', 'Momento_Aducao_Quadril_kg', 'Cisalhamento_ML_kg',
         'Compressao_Y_kg']
X = df[feats].to_numpy(float)
d = (df.Momento_Valgo_Pico_Nm / df.Massa_kg).to_numpy()
y = (d >= np.quantile(d, 0.75)).astype(int)
g = df.Participante.to_numpy()
print(f'Classe 1: {y.sum()} | P75 = {np.quantile(d, 0.75):.3f} Nm/kg')

def modelo():
    return make_pipeline(MinMaxScaler(), SVC(kernel='linear', C=1.0,
                                             class_weight='balanced', random_state=42))

for nome, fab, grp in (
        ('Por ciclo', lambda r: StratifiedKFold(3, shuffle=True, random_state=r), None),
        ('Por participante', lambda r: StratifiedGroupKFold(3, shuffle=True, random_state=r), g)):
    aucs, baccs, ignoradas = [], [], 0
    for r in range(50):
        s, pr = np.full(len(y), np.nan), np.full(len(y), -1)
        div = fab(r)
        for tr, te in (div.split(X, y, grp) if grp is not None else div.split(X, y)):
            if len(np.unique(y[tr])) < 2:
                ignoradas += 1
                continue
            m = modelo().fit(X[tr], y[tr])
            s[te], pr[te] = m.decision_function(X[te]), m.predict(X[te])
        ok = ~np.isnan(s)
        if len(np.unique(y[ok])) < 2:
            continue
        aucs.append(roc_auc_score(y[ok], s[ok]))
        baccs.append(balanced_accuracy_score(y[ok], pr[ok]))
    print(f'{nome}: AUC média {np.mean(aucs):.3f} [P2,5 {np.percentile(aucs, 2.5):.3f}; '
          f'P97,5 {np.percentile(aucs, 97.5):.3f}] | acurácia balanceada {np.mean(baccs):.3f} '
          f'| repetições válidas {len(aucs)}/50 | partições ignoradas {ignoradas}')

Classe 1: 8 | P75 = 0.154 Nm/kg
Por ciclo: AUC média 0.498 [P2,5 0.251; P97,5 0.673] | acurácia balanceada 0.493 | repetições válidas 50/50 | partições ignoradas 0
Por participante: AUC média 0.311 [P2,5 0.153; P97,5 0.471] | acurácia balanceada 0.417 | repetições válidas 50/50 | partições ignoradas 0
